# 🎯 02 — YOLOv8s Training Results Analysis
## Marine Biodiversity Ecosystem Assessment — Phase 3
**Shri Harsan M | M.Tech Data Science | SRM Institute**

This notebook covers:
- Training loss curves (box, cls, dfl)
- Validation metrics over epochs
- Precision-Recall curves per class
- Confusion matrix analysis
- Per-class mAP breakdown
- Speed benchmarks (FPS analysis)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from pathlib import Path

ROOT = Path('..').resolve()
RESULTS_DIR = ROOT / 'results' / 'detection' / 'yolov8s_marine'
CSV_PATH = RESULTS_DIR / 'results.csv'

print('Results dir:', RESULTS_DIR)
print('Exists:', RESULTS_DIR.exists())

## 1. Training Loss Curves

In [ ]:
if CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH)
    df.columns = df.columns.str.strip()
    print('Columns:', list(df.columns))
    print(f'Epochs: {len(df)}')
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('YOLOv8s Training Curves — Marine Biodiversity', fontsize=14, fontweight='bold')
    
    loss_cols  = ['train/box_loss', 'train/cls_loss', 'train/dfl_loss']
    val_cols   = ['val/box_loss',   'val/cls_loss',   'val/dfl_loss']
    titles     = ['Box Loss', 'Classification Loss', 'DFL Loss']
    colors_t   = ['#00c8ff', '#00ff9d', '#ffc200']
    colors_v   = ['#0090d4', '#00b870', '#e6a800']
    
    for i, (tc, vc, title) in enumerate(zip(loss_cols, val_cols, titles)):
        ax = axes[0, i]
        if tc in df: ax.plot(df[tc], label='Train', color=colors_t[i], linewidth=2)
        if vc in df: ax.plot(df[vc], label='Val',   color=colors_v[i], linewidth=2, linestyle='--')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.legend(); ax.grid(alpha=0.3)
    
    metric_cols = ['metrics/mAP50(B)', 'metrics/precision(B)', 'metrics/recall(B)']
    m_titles    = ['mAP@0.5', 'Precision', 'Recall']
    m_colors    = ['#ff3d5a', '#8b5cf6', '#f472b6']
    
    for i, (mc, mt, mc_) in enumerate(zip(metric_cols, m_titles, m_colors)):
        ax = axes[1, i]
        if mc in df:
            ax.plot(df[mc], color=mc_, linewidth=2)
            final = df[mc].iloc[-1]
            best  = df[mc].max()
            ax.axhline(best, color=mc_, linestyle=':', alpha=0.5, label=f'Best: {best:.4f}')
            ax.set_title(f'{mt} (final: {final:.4f})', fontweight='bold')
        ax.set_xlabel('Epoch'); ax.set_ylabel(mt)
        ax.legend(); ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../results/detection/yolov8s_marine/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('results.csv not found. Run training first.')
    print('Using known final values:')
    print('  mAP@0.5     = 0.951')
    print('  Precision   = 0.938')
    print('  Recall      = 0.928')
    print('  FPS         = 59.3')

## 2. Per-Class Performance Summary

In [ ]:
# Known results from Phase 3
results_table = {
    'Species':    ['Butterflyfish', 'Parrotfish', 'Angelfish', 'ALL (mAP)'],
    'Class':      [0, 1, 2, '-'],
    'Weight (w)': [2.0, 1.6, 1.1, '-'],
    'mAP@0.5':   [0.934, 0.967, 0.952, 0.951],
    'Precision':  [0.921, 0.952, 0.941, 0.938],
    'Recall':     [0.911, 0.941, 0.932, 0.928],
}

df_res = pd.DataFrame(results_table)
print(df_res.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
species = ['Butterflyfish', 'Parrotfish', 'Angelfish']
x = np.arange(len(species))
w = 0.25
ax.bar(x - w, [0.934, 0.967, 0.952], w, label='mAP@0.5', color='#00ff9d', alpha=0.85)
ax.bar(x,     [0.921, 0.952, 0.941], w, label='Precision', color='#00c8ff', alpha=0.85)
ax.bar(x + w, [0.911, 0.941, 0.932], w, label='Recall',    color='#8b5cf6', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(species)
ax.set_ylim(0.85, 1.0)
ax.set_ylabel('Score')
ax.set_title('Per-Class Detection Performance — YOLOv8s Marine', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.axhline(0.951, color='#ff3d5a', linestyle='--', alpha=0.7, label='Overall mAP')
plt.tight_layout()
plt.savefig('../results/detection/yolov8s_marine/per_class_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Speed Benchmark

In [ ]:
# Speed comparison: our system vs related works
works = {
    'Our System\n(YOLOv8s Marine)': {'fps': 59.3,  'map': 95.1, 'color': '#00ff9d'},
    'Qin 2024\n(YOLOv8-FASG)':    {'fps': 42.0,  'map': 96.4, 'color': '#00c8ff'},
    'Khiem 2025\n(PLOS ONE)':      {'fps': 28.5,  'map': 89.5, 'color': '#8b5cf6'},
    'Li 2023\n(Pattern Recog.)':   {'fps': 22.1,  'map': 87.3, 'color': '#ffc200'},
    'Zhao 2023\n(Ecol. Inform.)':  {'fps': 18.4,  'map': 84.6, 'color': '#ff3d5a'},
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Speed & Accuracy Comparison with Related Works', fontsize=13, fontweight='bold')

names  = list(works.keys())
fps    = [w['fps']  for w in works.values()]
maps   = [w['map']  for w in works.values()]
colors = [w['color']for w in works.values()]

bars1 = ax1.barh(names, fps, color=colors, alpha=0.85)
ax1.set_xlabel('FPS (frames per second)')
ax1.set_title('Inference Speed')
for bar, val in zip(bars1, fps):
    ax1.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2, f'{val}', va='center', fontweight='bold')

bars2 = ax2.barh(names, maps, color=colors, alpha=0.85)
ax2.set_xlabel('mAP@0.5 (%)')
ax2.set_title('Detection Accuracy')
ax2.set_xlim(80, 100)
for bar, val in zip(bars2, maps):
    ax2.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2, f'{val}%', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/detection/yolov8s_marine/speed_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nOur system achieves {59.3} FPS — {59.3/28.5:.1f}x faster than Khiem 2025')